In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector, RFE, SelectKBest, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import make_scorer, balanced_accuracy_score
from itertools import product
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

def get_sequential_methods():
    """
    Define sequential feature selection methods (fast ones only)
    
    Returns:
    --------
    methods : dict
        Dictionary of method name to (selector_class, params)
    """
    methods = {
        'SFS_forward': {
            'class': SequentialFeatureSelector,
            'params': {
                'direction': 'forward',
                'n_jobs': -1,
                'cv': 3
            }
        },
        'RFE': {
            'class': RFE,
            'params': {
                'step': 1
            }
        }
    }
    return methods

def get_base_estimators():
    """
    Get base estimators for sequential selection
    
    Returns:
    --------
    estimators : dict
        Dictionary of estimator name to estimator object
    """
    estimators = {
        'LogisticRegression': LogisticRegression(
            max_iter=1000, 
            class_weight='balanced', 
            random_state=RANDOM_STATE
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=50,  # Reduced for speed
            class_weight='balanced', 
            random_state=RANDOM_STATE,
            max_depth=10
        ),
        'GradientBoosting': GradientBoostingClassifier(
            n_estimators=50,  # Reduced for speed
            random_state=RANDOM_STATE,
            max_depth=5
        ),
        'SVC': SVC(
            class_weight='balanced',
            random_state=RANDOM_STATE,
            kernel='rbf'
        ),
        'KNN': KNeighborsClassifier(
            n_neighbors=5
        ),
        'MLP': MLPClassifier(
            hidden_layer_sizes=(50,),
            max_iter=500,
            random_state=RANDOM_STATE,
            early_stopping=True
        )
    }
    return estimators

def remove_correlated_features(X, threshold=0.7):
    """
    Remove highly correlated features
    
    Parameters:
    -----------
    X : DataFrame
        Feature matrix
    threshold : float
        Correlation threshold
    
    Returns:
    --------
    X_filtered : DataFrame
        Filtered feature matrix
    """
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = set()
    for col in upper.columns:
        high_corr = [i for i in upper.columns if upper.loc[i, col] > threshold]
        if high_corr:
            to_drop.update(high_corr)
    
    X_filtered = X.drop(columns=list(to_drop), errors='ignore')
    return X_filtered

def select_features_statistical(X, y, score_func, k=10):
    """
    Select features using statistical methods (SelectKBest)
    
    Parameters:
    -----------
    X : DataFrame
        Feature matrix
    y : Series
        Target vector
    score_func : callable
        Scoring function for SelectKBest
    k : int
        Number of top features to select
    
    Returns:
    --------
    selected_features : list
        List of selected feature names
    """
    skb = SelectKBest(score_func=score_func, k=min(k, X.shape[1]))
    skb.fit(X, y)
    selected_mask = skb.get_support()
    selected_features = X.columns[selected_mask].tolist()
    return selected_features

def evaluate_with_statistical_selection(X, y, score_func, n_features, 
                                        corr_threshold, estimator, cv_splits=3):
    """
    Evaluate model with statistical feature selection
    
    Parameters:
    -----------
    X : DataFrame
        Feature matrix
    y : Series
        Target vector
    score_func : callable
        Scoring function for SelectKBest
    n_features : int
        Number of features to select
    corr_threshold : float
        Correlation threshold for filtering
    estimator : estimator object
        Model to evaluate
    cv_splits : int
        Number of CV folds
    
    Returns:
    --------
    results : dict
        Dictionary with results
    """
    try:
        # Remove correlated features first
        X_filtered = remove_correlated_features(X, threshold=corr_threshold)
        
        # Select top k features
        selected_features = select_features_statistical(
            X_filtered, y, score_func, k=n_features
        )
        
        if len(selected_features) == 0:
            return {
                'accuracy_mean': np.nan,
                'accuracy_std': np.nan,
                'balanced_accuracy_mean': np.nan,
                'balanced_accuracy_std': np.nan,
                'n_features_selected': 0,
                'selected_features': ''
            }
        
        X_selected = X[selected_features]
        
        skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
        
        # Compute scores
        accuracy_scores = cross_val_score(
            estimator, X_selected, y, cv=skf, 
            scoring='accuracy', n_jobs=-1
        )
        balanced_accuracy_scores = cross_val_score(
            estimator, X_selected, y, cv=skf, 
            scoring='balanced_accuracy', n_jobs=-1
        )
        
        results = {
            'accuracy_mean': np.mean(accuracy_scores),
            'accuracy_std': np.std(accuracy_scores),
            'balanced_accuracy_mean': np.mean(balanced_accuracy_scores),
            'balanced_accuracy_std': np.std(balanced_accuracy_scores),
            'n_features_selected': len(selected_features),
            'selected_features': ','.join(selected_features[:10])
        }
        
        return results
        
    except Exception as e:
        print(f"Error in statistical selection: {e}")
        return {
            'accuracy_mean': np.nan,
            'accuracy_std': np.nan,
            'balanced_accuracy_mean': np.nan,
            'balanced_accuracy_std': np.nan,
            'n_features_selected': 0,
            'selected_features': ''
        }

def evaluate_with_sequential_selection(X, y, method_name, method_config, 
                                       estimator, n_features, corr_threshold, 
                                       cv_splits=3):
    """
    Evaluate model with sequential feature selection
    
    Parameters:
    -----------
    X : DataFrame
        Feature matrix
    y : Series
        Target vector
    method_name : str
        Name of sequential method
    method_config : dict
        Configuration for the selector
    estimator : estimator object
        Base estimator for selection
    n_features : int or str
        Number of features to select ('auto' for RFECV)
    cv_splits : int
        Number of CV folds for final evaluation
    
    Returns:
    --------
    results : dict
        Dictionary with results
    """
    try:
        # Remove correlated features first
        X_filtered = remove_correlated_features(X, threshold=corr_threshold)
        
        if X_filtered.shape[1] < n_features:
            n_features = max(2, X_filtered.shape[1])
        
        # Create selector
        selector_class = method_config['class']
        selector_params = method_config['params'].copy()
        
        # Handle n_features parameter
        if method_name == 'RFECV':
            # RFECV determines n_features automatically
            selector_params['min_features_to_select'] = max(2, n_features // 2)
        else:
            selector_params['n_features_to_select'] = n_features
        
        # Create selector with estimator
        selector = selector_class(estimator=estimator, **selector_params)
        
        # Fit selector on filtered features
        selector.fit(X_filtered, y)
        
        # Get selected features
        if hasattr(selector, 'get_support'):
            selected_mask = selector.get_support()
            selected_features = X_filtered.columns[selected_mask].tolist()
            n_selected = len(selected_features)
        else:
            selected_features = []
            n_selected = 0
        
        # Evaluate on selected features
        X_selected = X[selected_features]
        
        skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
        
        # Compute scores
        accuracy_scores = cross_val_score(
            estimator, X_selected, y, cv=skf, 
            scoring='accuracy', n_jobs=-1
        )
        balanced_accuracy_scores = cross_val_score(
            estimator, X_selected, y, cv=skf, 
            scoring='balanced_accuracy', n_jobs=-1
        )
        
        results = {
            'accuracy_mean': np.mean(accuracy_scores),
            'accuracy_std': np.std(accuracy_scores),
            'balanced_accuracy_mean': np.mean(balanced_accuracy_scores),
            'balanced_accuracy_std': np.std(balanced_accuracy_scores),
            'n_features_selected': n_selected,
            'selected_features': ','.join(selected_features[:10])  # Store first 10
        }
        
        return results
        
    except Exception as e:
        print(f"Error in {method_name}: {e}")
        return {
            'accuracy_mean': np.nan,
            'accuracy_std': np.nan,
            'balanced_accuracy_mean': np.nan,
            'balanced_accuracy_std': np.nan,
            'n_features_selected': 0,
            'selected_features': ''
        }

def run_sequential_search(stage_1_df, stage_2_df, target="is_unhealthy"):
    """
    Run comprehensive sequential feature selection search
    
    Parameters:
    -----------
    stage_1_df : DataFrame
        Stage 1 data
    stage_2_df : DataFrame
        Stage 2 data
    target : str
        Target column name
    
    Returns:
    --------
    results_df : DataFrame
        Complete results dataframe
    """
    # Base features (always included)
    base_features = [
        "hypertension", "early_rupture", "meconium", "induced", "no_progress",
        "is_male", "fetal_weight", "weeks", "is_first_birth", "age",
        "prev_birth", "normal_pres", "stage_1_percent", 
        "stage_1_normalized", "stage_2_normalized"
    ]

    # Signal processing methods
    preprocessing_methods = [
        "drop_long_then_autoencoder",
        "drop_long_then_interpolate",
        "drop_long_then_weighted_fill",
        "drop_zeros",
        "interpolate"
    ]

    # Sequential methods
    sequential_methods = get_sequential_methods()
    
    # Base estimators
    estimators = get_base_estimators()
    
    # Number of features to try
    n_features_options = [4, 6, 8, 10]
    
    # Correlation thresholds
    corr_thresholds = [0.5, 0.7, 0.9]
    
    # CV splits
    cv_splits = 3
    
    # Statistical score functions
    stat_score_funcs = {
        'f_classif': f_classif,
        'mutual_info': mutual_info_classif
    }
    
    all_results = []
    
    # Part 1: Statistical methods (fast)
    print("\n" + "="*80)
    print("PART 1: STATISTICAL FEATURE SELECTION (with correlation filtering)")
    print("="*80)
    
    stat_param_grid = {
        'stage': [('stage_1', stage_1_df), ('stage_2', stage_2_df)],
        'preprocessing': preprocessing_methods,
        'score_func': list(stat_score_funcs.keys()),
        'estimator': list(estimators.keys()),
        'n_features': n_features_options,
        'corr_threshold': corr_thresholds
    }
    
    total_stat = np.prod([len(v) for v in stat_param_grid.values()])
    print(f"Statistical configurations to test: {total_stat}")
    
    iteration = 0
    
    for (stage_name, df), prep_method, score_name, est_name, n_feat, corr_thresh in product(
        stat_param_grid['stage'],
        stat_param_grid['preprocessing'],
        stat_param_grid['score_func'],
        stat_param_grid['estimator'],
        stat_param_grid['n_features'],
        stat_param_grid['corr_threshold']
    ):
        iteration += 1
        if iteration % 50 == 0:
            print(f"Progress: {iteration}/{total_stat} ({100*iteration/total_stat:.1f}%)")
        
        try:
            # Get method-specific features
            method_features = base_features + [
                col for col in df.columns if col.endswith(prep_method)
            ]
            
            X = df[method_features]
            y = df[target]
            
            # Get score function and estimator
            score_func = stat_score_funcs[score_name]
            estimator = estimators[est_name]
            
            # Evaluate
            results = evaluate_with_statistical_selection(
                X, y, score_func, n_feat, corr_thresh, estimator, cv_splits
            )
            
            # Store results
            row = {
                'stage': stage_name,
                'preprocessing_method': prep_method,
                'selection_method': f'Statistical_{score_name}',
                'base_estimator': est_name,
                'n_features_requested': n_feat,
                'corr_threshold': corr_thresh,
                'cv_folds': cv_splits
            }
            row.update(results)
            all_results.append(row)
            
        except Exception as e:
            print(f"Error: {stage_name}/{prep_method}/{score_name}/{est_name}: {e}")
            continue
    
    # Part 2: Sequential methods
    print("\n" + "="*80)
    print("PART 2: SEQUENTIAL FEATURE SELECTION (with correlation filtering)")
    print("="*80)
    
    seq_param_grid = {
        'stage': [('stage_1', stage_1_df), ('stage_2', stage_2_df)],
        'preprocessing': preprocessing_methods,
        'sequential_method': list(sequential_methods.keys()),
        'estimator': list(estimators.keys()),
        'n_features': n_features_options,
        'corr_threshold': corr_thresholds
    }
    
    total_seq = np.prod([len(v) for v in seq_param_grid.values()])
    print(f"Sequential configurations to test: {total_seq}")
    
    iteration = 0
    
    for (stage_name, df), prep_method, seq_method_name, est_name, n_feat, corr_thresh in product(
        seq_param_grid['stage'],
        seq_param_grid['preprocessing'],
        seq_param_grid['sequential_method'],
        seq_param_grid['estimator'],
        seq_param_grid['n_features'],
        seq_param_grid['corr_threshold']
    ):
        iteration += 1
        if iteration % 20 == 0:
            print(f"Progress: {iteration}/{total_seq} ({100*iteration/total_seq:.1f}%)")

        # Skip incompatible RFE combinations
        if seq_method_name == 'RFE' and est_name in ['SVC', 'KNN', 'MLP']:
            continue
            
        try:
            # Get method-specific features
            method_features = base_features + [
                col for col in df.columns if col.endswith(prep_method)
            ]
            
            # Skip if not enough features
            if len(method_features) < n_feat:
                continue
            
            X = df[method_features]
            y = df[target]
            
            # Get sequential method config and estimator
            seq_method_config = sequential_methods[seq_method_name]
            estimator = estimators[est_name]
            
            # Evaluate
            results = evaluate_with_sequential_selection(
                X, y, seq_method_name, seq_method_config, 
                estimator, n_feat, corr_thresh, cv_splits
            )
            
            # Store results
            row = {
                'stage': stage_name,
                'preprocessing_method': prep_method,
                'selection_method': seq_method_name,
                'base_estimator': est_name,
                'n_features_requested': n_feat,
                'corr_threshold': corr_thresh,
                'cv_folds': cv_splits
            }
            row.update(results)
            all_results.append(row)
            
        except Exception as e:
            print(f"Error: {stage_name}/{prep_method}/{seq_method_name}/{est_name}: {e}")
            continue
    
    results_df = pd.DataFrame(all_results)
    return results_df

# Example usage
if __name__ == "__main__":
    # Load data
    stage_1_df = pd.read_csv("data/stage_1.csv", index_col=0)
    stage_2_df = pd.read_csv("data/stage_2.csv", index_col=0)
    
    print(f"Baseline (majority class): {1 - stage_1_df['is_unhealthy'].mean():.4f}")
    print(f"Target to beat: 0.72\n")
    
    # Run sequential search
    results_df = run_sequential_search(stage_1_df, stage_2_df)
    
    # Save results
    results_df.to_csv("sequential_results.csv", index=False)
    
    # Display top configurations
    print("\n" + "="*100)
    print("TOP 20 CONFIGURATIONS BY BALANCED ACCURACY")
    print("="*100)
    top_configs = results_df.nlargest(20, 'balanced_accuracy_mean')[
        ['stage', 'preprocessing_method', 'selection_method', 'base_estimator',
         'n_features_requested', 'n_features_selected', 'accuracy_mean', 
         'balanced_accuracy_mean', 'balanced_accuracy_std']
    ]
    print(top_configs.to_string(index=False))
    
    # Show which configs beat 0.72
    print("\n" + "="*100)
    print("CONFIGURATIONS BEATING 0.72 ACCURACY")
    print("="*100)
    beat_threshold = results_df[results_df['accuracy_mean'] > 0.72].sort_values(
        'accuracy_mean', ascending=False
    )
    if len(beat_threshold) > 0:
        print(beat_threshold[
            ['stage', 'preprocessing_method', 'selection_method', 'base_estimator',
             'accuracy_mean', 'balanced_accuracy_mean', 'n_features_selected']
        ].to_string(index=False))
    else:
        print("No configurations beat 0.72 accuracy threshold")
    
    # Summary statistics
    print("\n" + "="*100)
    print("SUMMARY BY SELECTION METHOD")
    print("="*100)
    print(results_df.groupby('selection_method').agg({
        'accuracy_mean': ['mean', 'std', 'max'],
        'balanced_accuracy_mean': ['mean', 'std', 'max']
    }).round(4))
    
    print("\n" + "="*100)
    print("SUMMARY BY BASE ESTIMATOR")
    print("="*100)
    print(results_df.groupby('base_estimator').agg({
        'accuracy_mean': ['mean', 'std', 'max'],
        'balanced_accuracy_mean': ['mean', 'std', 'max']
    }).round(4))
    
    print("\n" + "="*100)
    print("SUMMARY BY PREPROCESSING METHOD")
    print("="*100)
    print(results_df.groupby('preprocessing_method').agg({
        'accuracy_mean': ['mean', 'std', 'max'],
        'balanced_accuracy_mean': ['mean', 'std', 'max']
    }).round(4))

Baseline (majority class): 0.6491
Target to beat: 0.72


PART 1: STATISTICAL FEATURE SELECTION (with correlation filtering)
Statistical configurations to test: 1440
Progress: 50/1440 (3.5%)
Progress: 100/1440 (6.9%)
Progress: 150/1440 (10.4%)
Progress: 200/1440 (13.9%)
Progress: 250/1440 (17.4%)
Progress: 300/1440 (20.8%)
Progress: 350/1440 (24.3%)
Progress: 400/1440 (27.8%)
Progress: 450/1440 (31.2%)
Progress: 500/1440 (34.7%)
Progress: 550/1440 (38.2%)
Progress: 600/1440 (41.7%)
Progress: 650/1440 (45.1%)
Progress: 700/1440 (48.6%)
Progress: 750/1440 (52.1%)
Progress: 800/1440 (55.6%)
Progress: 850/1440 (59.0%)
Progress: 900/1440 (62.5%)
Progress: 950/1440 (66.0%)
Progress: 1000/1440 (69.4%)
Progress: 1050/1440 (72.9%)
Progress: 1100/1440 (76.4%)
Progress: 1150/1440 (79.9%)
Progress: 1200/1440 (83.3%)
Progress: 1250/1440 (86.8%)
Progress: 1300/1440 (90.3%)
Progress: 1350/1440 (93.8%)
Progress: 1400/1440 (97.2%)

PART 2: SEQUENTIAL FEATURE SELECTION (with correlation filtering)
Sequen

In [27]:
results_df["accuracy_mean"].max()

np.float64(0.7105263157894738)

In [30]:
results_df["accuracy_mean"]


0       0.596491
1       0.614035
2       0.614035
3       0.587719
4       0.592105
          ...   
2875         NaN
2876         NaN
2877         NaN
2878         NaN
2879         NaN
Name: accuracy_mean, Length: 2880, dtype: float64